Predikce hodinové spotřeby elektřiny jedné domácnosti na základě času, předchozí spotřeby a venkovní teploty s využitím datasetu "Individual Household Electric Power Consumption in Sceaux, Paris"

# Setup

In [7]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler 
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

SEED = 20261017

def stream_seed(stream_id, repetition=0):
    return int(np.random.SeedSequence([SEED, stream_id, repetition]).generate_state(1)[0])


DATA = Path("data")
assert (DATA / "sceaux_hourly_temperature.csv").is_file(), "Temperature file not loaded"


FEATURES_A = ["temperature_2m"]

TARGET = "Global_active_power"

# Datasets

**Household consumption** ,easurements of electric power consumption in one household with a one-minute sampling rate in Sceaux (7km of Paris, France) between December 2006 and November 2010 (47 months).
Requires `pip install ucimlrepo`

Source: https://archive.ics.uci.edu/dataset/235/individual+household+electric+power+consumption

In [19]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
dataset = fetch_ucirepo(id=235) 
  
# data (as pandas dataframes) 
df_consumption = dataset.data.original.replace(["", "?"], pd.NA)
print(df_consumption.head())


/Users/angelina/UHK/Masters Study/UHK2_Sem3/SU/ukol1/.venv/lib/python3.14/site-packages/ucimlrepo/fetch.py:97: DtypeWarning: Columns (2: Global_active_power, 3: Global_reactive_power, 4: Voltage, 5: Global_intensity, 6: Sub_metering_1, 7: Sub_metering_2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_url)


         Date      Time Global_active_power Global_reactive_power  Voltage  \
0  16/12/2006  17:24:00               4.216                 0.418  234.840   
1  16/12/2006  17:25:00               5.360                 0.436  233.630   
2  16/12/2006  17:26:00               5.374                 0.498  233.290   
3  16/12/2006  17:27:00               5.388                 0.502  233.740   
4  16/12/2006  17:28:00               3.666                 0.528  235.680   

  Global_intensity Sub_metering_1 Sub_metering_2  Sub_metering_3  
0           18.400          0.000          1.000            17.0  
1           23.000          0.000          1.000            16.0  
2           23.000          0.000          2.000            17.0  
3           23.000          0.000          1.000            17.0  
4           15.800          0.000          1.000            17.0  


**Temperature dataset** downloaded from Open Meteo with per-hour historic temperature from the area of Sceaux.
- latitude 48.7764, longitude 2.2903
- weather model ERA5-Land
- from 01.12.2006 until 30.11.2010
- timezone UTC

Data source: https://open-meteo.com/en/docs/historical-weather-api?utm_source=chatgpt.com&bounding_box=-90,-180,90,180

In [10]:
df_temperature_metadata = pd.read_csv(DATA / "sceaux_temperature_metadata.csv")
print(df_temperature_metadata.head())

df_temperature = pd.read_csv(DATA / "sceaux_hourly_temperature.csv")
print(df_temperature.head())

    latitude  longitude  elevation  utc_offset_seconds timezone  \
0  48.800003   2.300003       94.0                   0      GMT   

  timezone_abbreviation  
0                   GMT  
               time  temperature_2m (°C)
0  2006-12-01T00:00                  6.5
1  2006-12-01T01:00                  6.1
2  2006-12-01T02:00                  5.9
3  2006-12-01T03:00                  5.7
4  2006-12-01T04:00                  5.4


# EDA

In [11]:
# helpers
def audit(frame, unique_features):
    display(frame.head())
    print("Size:", frame.shape)
    display(pd.DataFrame({"type": frame.dtypes, "missing": frame.isna().sum()}))
    print("Exact duplicates:", frame.duplicated().sum())
    
    for feature in unique_features:
        assert frame[feature].is_unique, print("Duplicated:", frame[feature].duplicated().sum())
    print("Verified unique features:", unique_features)

def audit_complete_frame(frame, features, target):    
    assert frame[features + [target]].notna().all().all()
    assert np.isfinite(frame[features + [target]].to_numpy(dtype=float)).all()

def scores(y, prediction):
    return {"MAE [kWh]": mean_absolute_error(y, prediction),
            "RMSE [kWh]": root_mean_squared_error(y, prediction)}

## Basic audit of datasets

In [12]:
audit(df_temperature, ["time"])

,time,temperature_2m (°C)
0,2006-12-01T00:00,6.5
1,2006-12-01T01:00,6.1
2,2006-12-01T02:00,5.9
3,2006-12-01T03:00,5.7
4,2006-12-01T04:00,5.4


Size: (35064, 2)


,type,missing
time,str,0
temperature_2m (°C),float64,0


Exact duplicates: 0
Verified unique features: ['time']


In [20]:
audit(df_consumption, [])

,Date,Time,Global_active_power,Global_reactive_power,Voltage,Global_intensity,Sub_metering_1,Sub_metering_2,Sub_metering_3
0,16/12/2006,17:24:00,4.216,0.418,234.840,18.400,0.000,1.000,17.0
1,16/12/2006,17:25:00,5.360,0.436,233.630,23.000,0.000,1.000,16.0
2,16/12/2006,17:26:00,5.374,0.498,233.290,23.000,0.000,2.000,17.0
3,16/12/2006,17:27:00,5.388,0.502,233.740,23.000,0.000,1.000,17.0
4,16/12/2006,17:28:00,3.666,0.528,235.680,15.800,0.000,1.000,17.0


Size: (2075259, 9)


,type,missing
Date,str,0
Time,str,0
Global_active_power,object,25979
Global_reactive_power,object,25979
Voltage,object,25979
Global_intensity,object,25979
Sub_metering_1,object,25979
Sub_metering_2,object,25979
Sub_metering_3,float64,25979


Exact duplicates: 0
Verified unique features: []


Output:
* **Temperature dataset** 
    * contains no duplicates and missing values
    * types do not require the transformation: time is string and temperature is float
    * time is verified to be unique across the whole dataset
    * UTC timezone (from metadata)
* **Consumption dataset** 
    * contains no duplicates
    * point of interest is 'Global_active_power', other features may be dropped
    * 'Global_active_power' has 25979 missing values
    * 'Global_active_power' and other measurements are of type object, requires the transformation to float
    * the measurements are per minute, while the temperature data is per hour
    * transformation to the kWh will be applied on 'Global_active_power', along with that the missing values will be handled.
    * 'Date' and 'Time' columns will be merged to easily map to 'time' form temperature dataset after the transformation to per-hour measurements
    * NOTE: the timezone is not clear from the datasource, therefore UTC is assumed

## Data pre-processing

### Household consumption dataset
1. limit only to 'Date', 'Time' and 'Global_active_power'
2. transform the 'Global_active_power' from object to float64
3. merge 'Date' and 'Time' columns into one in format similar to 'time' from temeprature data "yyyy-mm-ddThh:mm" as a start hour of a one-hour timestamp
4. transform the per-minute time series to per-hour 
    - SUM(Global_active_power in 60 minutes per hour) / 60
    - before that, check how many missing values in scope of one hour
        - count valid measurements for each hour
        - keep hours with 60 valid minute measurements
        - mark hours with missing values with NA for a start

In [ ]:
# 1. Limit only to 'Date', 'Time' and 'Global_active_power'.
# Consumption dataset contains additional fields, 
# but the point of interest is `Global_active_power`, so other columns will be dropped.
df_consumption_target = df_consumption[["Date", "Time", "Global_active_power"]]
print(df_consumption_target.head())

         Date      Time  Global_active_power
0  16/12/2006  17:24:00                4.216
1  16/12/2006  17:25:00                5.360
2  16/12/2006  17:26:00                5.374
3  16/12/2006  17:27:00                5.388
4  16/12/2006  17:28:00                3.666


In [ ]:
# 2. transform the 'Global_active_power' from object to float64
numeric_cols = ["Global_active_power"]

df_consumption_target[numeric_cols] = (
    df_consumption_target[numeric_cols]
    .apply(pd.to_numeric, errors="coerce")
    .astype("float64")
)

print(df_consumption_target.dtypes)

Date                       str
Time                       str
Global_active_power    float64
dtype: object


In [40]:
# 3. Merge Date and Time into the start of each one-hour interval.
df_consumption_target_time = df_consumption_target.copy()
df_consumption_target_time["time"] = (
    pd.to_datetime(
        df_consumption_target["Date"] + " " + df_consumption_target["Time"],
        format="%d/%m/%Y %H:%M:%S",
    )
    .dt.floor("h")
)
df_consumption_target_time = df_consumption_target_time.drop(columns=["Date", "Time"])

print(df_consumption_target_time.head())
print(df_consumption_target_time.dtypes)

   Global_active_power                time
0                4.216 2006-12-16 17:00:00
1                5.360 2006-12-16 17:00:00
2                5.374 2006-12-16 17:00:00
3                5.388 2006-12-16 17:00:00
4                3.666 2006-12-16 17:00:00
Global_active_power           float64
time                   datetime64[us]
dtype: object


In [ ]:
# Transform
df_consumption_transformed = None

# Count valid and missing measurements for each hour before transformation.
# size counts all rows, count takes only non-missing
hourly_measurement_counts = (
    df_consumption_target_time.groupby("time")["Global_active_power"]
    .agg(total_values="size", valid_values="count")
)
hourly_measurement_counts["missing_values"] = (
    hourly_measurement_counts["total_values"]
    - hourly_measurement_counts["valid_values"]
)

# Display hours containing at least one missing measurement.
display(
    hourly_measurement_counts.loc[
        hourly_measurement_counts["missing_values"] > 0,
        ["valid_values", "missing_values"],
    ].reset_index()
)

,time,valid_values,missing_values
0,2006-12-21T11:00,58,2
1,2006-12-30T10:00,58,2
2,2007-01-14T18:00,59,1
3,2007-01-28T17:00,59,1
4,2007-02-22T22:00,58,2
...,...,...,...
497,2010-09-28T16:00,0,60
498,2010-09-28T17:00,0,60
499,2010-09-28T18:00,0,60
500,2010-09-28T19:00,47,13


Missing values: some hours miss only 1-2 values, some have no valid value. 

- Target consumption missing: exclude in all cases
- Consumption 24 hours earlier is missing: excluded if previous consumption is a feature

Steps:
1. Aggregate to hourly consumption. 
    - Calculate sum / 60 only for hours with 60 valid minute measurements. 
    - Otherwise, set hourly consumption to NaN. Preserve a regular hourly timeline, including those NaN hours.
3. Create yesterday’s consumption feature: consumption exactly 24 hours earlier.
4. Join temperature data using aligned timestamps.
5. Select usable rows for each model.

In [41]:
# 1. Aggregate complete hours to energy consumption in kWh.
minute_consumption = df_consumption_target_time.assign(
    time=pd.to_datetime(df_consumption_target_time["time"], format="%Y-%m-%dT%H:%M")
).set_index("time")["Global_active_power"]

# Resampling also inserts absent hours, preserving the hourly timeline.
hourly_consumption = minute_consumption.resample("h")
valid_measurements = hourly_consumption.count()
df_consumption_transformed = (
    (hourly_consumption.sum(min_count=60) / 60)
    .where(valid_measurements.eq(60))
    .rename("Global_active_power")
    .reset_index()
)
df_consumption_transformed["time"] = (
    df_consumption_transformed["time"].dt.strftime("%Y-%m-%dT%H:%M")
)
df_consumption_hourly = df_consumption_transformed.copy()

# Incomplete and absent hours must have missing consumption.
assert df_consumption_hourly["Global_active_power"].isna().to_numpy()[
    valid_measurements.ne(60).to_numpy()
].all()
display(df_consumption_hourly.head())


,time,Global_active_power
0,2006-12-16T17:00,NaN
1,2006-12-16T18:00,3.632200
2,2006-12-16T19:00,3.400233
3,2006-12-16T20:00,3.268567
4,2006-12-16T21:00,3.056467


## Merge datasets

### Data range check

In [10]:
# Date range comparison TODO after date transformation on df_consumption
print("Temperature time range:", df_temperature.time.min(), "-", df_temperature.time.max())
print("Consumption time range:", df_consumption.Date.min(), "-", df_consumption.Date.max())

Temperature time range: 2006-12-01T00:00 - 2010-11-30T23:00
Consumption time range: 1/1/2007 00:00:00 - 9/9/2010 23:59:00
